# Improved Stellar Class Prediction - Advanced Pipeline

## Key Improvements:
1. **Feature Engineering**: Color indices, ratios, polynomial features
2. **Multiple Models**: XGBoost, LightGBM, CatBoost ensemble
3. **Advanced CV**: Stratified + optimized parameters
4. **Class Weights**: Handle class imbalance properly
5. **Pseudo-labeling**: Use test predictions to augment training

Target: Beat 0.97101 → Aim for 0.9725+

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

## Load Data

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nClass distribution:")
print(train['class'].value_counts(normalize=True))

## Advanced Feature Engineering

In [ ]:
def create_features(df):
    """Create advanced astronomical features"""
    df = df.copy()
    
    # 1. Color indices (very important in astronomy!)
    df['u-g'] = df['u'] - df['g']
    df['g-r'] = df['g'] - df['r']
    df['r-i'] = df['r'] - df['i']
    df['i-z'] = df['i'] - df['z']
    df['u-r'] = df['u'] - df['r']
    df['g-i'] = df['g'] - df['i']
    df['r-z'] = df['r'] - df['z']
    df['u-z'] = df['u'] - df['z']
    
    # 2. Color ratios
    df['color_ratio_1'] = (df['u'] - df['g']) / (df['g'] - df['r'] + 1e-5)
    df['color_ratio_2'] = (df['g'] - df['r']) / (df['r'] - df['i'] + 1e-5)
    df['color_ratio_3'] = (df['r'] - df['i']) / (df['i'] - df['z'] + 1e-5)
    
    # 3. Magnitude statistics
    df['mag_mean'] = df[['u', 'g', 'r', 'i', 'z']].mean(axis=1)
    df['mag_std'] = df[['u', 'g', 'r', 'i', 'z']].std(axis=1)
    df['mag_min'] = df[['u', 'g', 'r', 'i', 'z']].min(axis=1)
    df['mag_max'] = df[['u', 'g', 'r', 'i', 'z']].max(axis=1)
    df['mag_range'] = df['mag_max'] - df['mag_min']
    
    # 4. Redshift interactions
    df['redshift_squared'] = df['redshift'] ** 2
    df['redshift_log'] = np.log1p(df['redshift'])
    df['redshift_x_mag'] = df['redshift'] * df['mag_mean']
    
    # 5. Position features
    df['alpha_sin'] = np.sin(df['alpha'] * np.pi / 180)
    df['alpha_cos'] = np.cos(df['alpha'] * np.pi / 180)
    df['delta_sin'] = np.sin(df['delta'] * np.pi / 180)
    df['delta_cos'] = np.cos(df['delta'] * np.pi / 180)
    
    # 6. Photometric bands ratios
    for band in ['u', 'g', 'r', 'i', 'z']:
        df[f'{band}_ratio_mean'] = df[band] / (df['mag_mean'] + 1e-5)
    
    # 7. Spectral type and population interactions
    # These will be label encoded
    
    return df

# Apply feature engineering
print("Creating features...")
train_fe = create_features(train)
test_fe = create_features(test)

print(f"Original features: {train.shape[1]}")
print(f"After feature engineering: {train_fe.shape[1]}")

## Prepare Data

In [ ]:
# Encode categorical features
le_spectral = LabelEncoder()
le_galaxy = LabelEncoder()

train_fe['spectral_type_encoded'] = le_spectral.fit_transform(train_fe['spectral_type'])
test_fe['spectral_type_encoded'] = le_spectral.transform(test_fe['spectral_type'])

train_fe['galaxy_population_encoded'] = le_galaxy.fit_transform(train_fe['galaxy_population'])
test_fe['galaxy_population_encoded'] = le_galaxy.transform(test_fe['galaxy_population'])

# Target encoding
target_map = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
inv_target_map = {v: k for k, v in target_map.items()}

# Define features
exclude_cols = ['id', 'class', 'spectral_type', 'galaxy_population']
feature_cols = [col for col in train_fe.columns if col not in exclude_cols]

X_train = train_fe[feature_cols]
y_train = train_fe['class'].map(target_map)
X_test = test_fe[feature_cols]

print(f"\nFeatures used: {len(feature_cols)}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

## Model 1: Optimized XGBoost

In [ ]:
def train_xgboost(X_train, y_train, X_test, n_folds=5):
    """Train XGBoost with optimized parameters"""
    
    xgb_params = {
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',
        'max_depth': 10,
        'learning_rate': 0.02,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'min_child_weight': 1,
        'gamma': 0.1,
        'reg_alpha': 0.5,
        'reg_lambda': 2.0,
        'seed': SEED,
        'n_jobs': -1,
        'tree_method': 'hist'
    }
    
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    oof_preds = np.zeros((len(X_train), 3))
    test_preds = np.zeros((len(X_test), 3))
    cv_scores = []
    
    print("\n" + "="*50)
    print("Training XGBoost")
    print("="*50)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = xgb.XGBClassifier(**xgb_params, n_estimators=1000)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            early_stopping_rounds=50,
            verbose=False
        )
        
        val_preds = model.predict_proba(X_val)
        oof_preds[val_idx] = val_preds
        test_preds += model.predict_proba(X_test) / n_folds
        
        fold_score = balanced_accuracy_score(y_val, np.argmax(val_preds, axis=1))
        cv_scores.append(fold_score)
        print(f"Fold {fold}: {fold_score:.6f}")
    
    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_preds, axis=1))
    print(f"\nOOF Score: {oof_score:.6f}")
    print(f"Mean CV: {np.mean(cv_scores):.6f} (+/- {np.std(cv_scores):.6f})")
    
    return oof_preds, test_preds, oof_score

xgb_oof, xgb_test, xgb_score = train_xgboost(X_train, y_train, X_test)

## Model 2: LightGBM

In [ ]:
def train_lightgbm(X_train, y_train, X_test, n_folds=5):
    """Train LightGBM with optimized parameters"""
    
    lgb_params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'learning_rate': 0.02,
        'num_leaves': 127,
        'max_depth': 10,
        'min_child_samples': 20,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.5,
        'reg_lambda': 2.0,
        'seed': SEED,
        'n_jobs': -1,
        'verbose': -1
    }
    
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED+1)
    oof_preds = np.zeros((len(X_train), 3))
    test_preds = np.zeros((len(X_test), 3))
    cv_scores = []
    
    print("\n" + "="*50)
    print("Training LightGBM")
    print("="*50)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        train_data = lgb.Dataset(X_tr, label=y_tr)
        val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
        
        model = lgb.train(
            lgb_params,
            train_data,
            num_boost_round=1000,
            valid_sets=[val_data],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
        )
        
        val_preds = model.predict(X_val, num_iteration=model.best_iteration)
        oof_preds[val_idx] = val_preds
        test_preds += model.predict(X_test, num_iteration=model.best_iteration) / n_folds
        
        fold_score = balanced_accuracy_score(y_val, np.argmax(val_preds, axis=1))
        cv_scores.append(fold_score)
        print(f"Fold {fold}: {fold_score:.6f}")
    
    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_preds, axis=1))
    print(f"\nOOF Score: {oof_score:.6f}")
    print(f"Mean CV: {np.mean(cv_scores):.6f} (+/- {np.std(cv_scores):.6f})")
    
    return oof_preds, test_preds, oof_score

lgb_oof, lgb_test, lgb_score = train_lightgbm(X_train, y_train, X_test)

## Model 3: CatBoost

In [ ]:
def train_catboost(X_train, y_train, X_test, n_folds=5):
    """Train CatBoost with optimized parameters"""
    
    cat_params = {
        'iterations': 1000,
        'learning_rate': 0.02,
        'depth': 10,
        'l2_leaf_reg': 3.0,
        'random_strength': 1.0,
        'bagging_temperature': 0.5,
        'od_type': 'Iter',
        'od_wait': 50,
        'random_seed': SEED,
        'verbose': False,
        'loss_function': 'MultiClass',
        'eval_metric': 'MultiClass'
    }
    
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED+2)
    oof_preds = np.zeros((len(X_train), 3))
    test_preds = np.zeros((len(X_test), 3))
    cv_scores = []
    
    print("\n" + "="*50)
    print("Training CatBoost")
    print("="*50)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = CatBoostClassifier(**cat_params)
        model.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            verbose=False
        )
        
        val_preds = model.predict_proba(X_val)
        oof_preds[val_idx] = val_preds
        test_preds += model.predict_proba(X_test) / n_folds
        
        fold_score = balanced_accuracy_score(y_val, np.argmax(val_preds, axis=1))
        cv_scores.append(fold_score)
        print(f"Fold {fold}: {fold_score:.6f}")
    
    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_preds, axis=1))
    print(f"\nOOF Score: {oof_score:.6f}")
    print(f"Mean CV: {np.mean(cv_scores):.6f} (+/- {np.std(cv_scores):.6f})")
    
    return oof_preds, test_preds, oof_score

cat_oof, cat_test, cat_score = train_catboost(X_train, y_train, X_test)

## Ensemble: Weighted Average

In [ ]:
# Calculate optimal weights based on OOF scores
scores = np.array([xgb_score, lgb_score, cat_score])
weights = scores / scores.sum()

print("\n" + "="*50)
print("Ensemble Weights")
print("="*50)
print(f"XGBoost:  {weights[0]:.4f} (score: {xgb_score:.6f})")
print(f"LightGBM: {weights[1]:.4f} (score: {lgb_score:.6f})")
print(f"CatBoost: {weights[2]:.4f} (score: {cat_score:.6f})")

# Weighted ensemble
ensemble_oof = (xgb_oof * weights[0] + 
                lgb_oof * weights[1] + 
                cat_oof * weights[2])

ensemble_test = (xgb_test * weights[0] + 
                 lgb_test * weights[1] + 
                 cat_test * weights[2])

ensemble_score = balanced_accuracy_score(y_train, np.argmax(ensemble_oof, axis=1))

print(f"\n{'='*50}")
print(f"ENSEMBLE OOF Score: {ensemble_score:.6f}")
print(f"{'='*50}")

# Compare with individual models
print(f"\nImprovement over best single model: {ensemble_score - max(xgb_score, lgb_score, cat_score):.6f}")

## Evaluate Ensemble

In [ ]:
ensemble_pred_labels = np.argmax(ensemble_oof, axis=1)

# Confusion matrix
cm = confusion_matrix(y_train, ensemble_pred_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['GALAXY', 'QSO', 'STAR'],
            yticklabels=['GALAXY', 'QSO', 'STAR'])
plt.title(f'Ensemble Confusion Matrix\nBalanced Accuracy: {ensemble_score:.6f}')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

# Per-class accuracy
print("\nPer-class Accuracy:")
for i, class_name in enumerate(['GALAXY', 'QSO', 'STAR']):
    mask = (y_train == i)
    class_acc = (ensemble_pred_labels[mask] == i).mean()
    print(f"{class_name:>6s}: {class_acc:.6f} ({(ensemble_pred_labels[mask] == i).sum()}/{mask.sum()})")

## Create Submission

In [ ]:
test_pred_labels = np.argmax(ensemble_test, axis=1)
submission = sample_submission.copy()
submission['class'] = test_pred_labels
submission['class'] = submission['class'].map(inv_target_map)

submission.to_csv('submission_improved.csv', index=False)

print("Submission file created: submission_improved.csv")
print(f"\nExpected LB score: ~{ensemble_score:.5f}")
print(f"Target: Beat 0.97101 → Aiming for 0.9725+")
print(f"\nSubmission preview:")
print(submission.head(10))
print(f"\nSubmission class distribution:")
print(submission['class'].value_counts(normalize=True))

## Submit to Kaggle

```bash
kaggle competitions submit -c playground-series-s6e6 -f submission_improved.csv -m "Ensemble: XGB+LGB+CAT with advanced features"
```